In [1]:
!pip install segmentation_models_pytorch -q
import os
import json
import cv2
import torch
import numpy as np
import torchvision
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.mask_rcnn import MaskRCNNPredictor
from tqdm import tqdm
import albumentations as A
from albumentations.pytorch import ToTensorV2
import gc

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"🚀 Uniformed Mask R-CNN (SCRATCH) Initialized on: {device}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 3.7 MB/s eta 0:00:00
🚀 Uniformed Mask R-CNN (SCRATCH) Initialized on: cuda


In [2]:
IMG_SIZE = 640
BATCH_SIZE = 4
EPOCHS = 5      # Increased slightly from 10 to 15 to help scratch convergence
FRACTION = 0.25 
NUM_CLASSES = 6 

LABEL_MAP = {
    "background": 0, "short sleeve top": 1, "trousers": 2, 
    "shorts": 3, "long sleeve top": 4, "skirt": 5
}
INV_LABEL_MAP = {v: k for k, v in LABEL_MAP.items()}

In [3]:
class FashionInstanceDataset(Dataset):
    def __init__(self, split='train', transform=None):
        self.base_dir = f"/kaggle/input/datasets/shubhranilbasak/vr-trimmed-dataset/Trimmed-vr-dataset/DeepFashion2_Top5_{split}"
        self.img_dir = os.path.join(self.base_dir, "images")
        self.anno_dir = os.path.join(self.base_dir, "annos")
        self.filenames = [f.replace('.json', '') for f in os.listdir(self.anno_dir) if f.endswith('.json')]
        self.transform = transform

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        img = cv2.cvtColor(cv2.imread(os.path.join(self.img_dir, f"{name}.jpg")), cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        with open(os.path.join(self.anno_dir, f"{name}.json"), 'r') as f:
            data = json.load(f)
        
        masks, boxes, labels = [], [], []
        for key, item in data.items():
            if key.startswith('item') and item.get('category_name') in LABEL_MAP:
                m = np.zeros((h, w), dtype=np.uint8)
                for poly in item.get('segmentation', []):
                    cv2.fillPoly(m, [np.array(poly).reshape(-1, 2).astype(np.int32)], 1)
                
                x, y, x2, y2 = item.get('bounding_box', [0,0,0,0])
                # Clipping for stability
                x, y, x2, y2 = max(0, x), max(0, y), min(w, x2), min(h, y2)
                
                if (x2 - x) > 1 and (y2 - y) > 1:
                    masks.append(m)
                    boxes.append([x, y, x2, y2])
                    labels.append(LABEL_MAP[item.get('category_name')])

        if not labels: # Fallback for empty images
            masks, boxes, labels = [np.zeros((h, w), dtype=np.uint8)], [[0, 0, 10, 10]], [0]

        if self.transform:
            augmented = self.transform(image=img, masks=masks, bboxes=boxes, category_ids=labels)
            img, masks = augmented['image'], torch.as_tensor(np.stack(augmented['masks']), dtype=torch.uint8)
            boxes, labels = torch.as_tensor(augmented['bboxes'], dtype=torch.float32), torch.as_tensor(augmented['category_ids'], dtype=torch.int64)
        
        return img, {"boxes": boxes, "labels": labels, "masks": masks}

def collate_fn(batch): return tuple(zip(*batch))

# Transformers
train_tf = A.Compose([A.Resize(IMG_SIZE, IMG_SIZE), A.Normalize(), ToTensorV2()], 
                     bbox_params=A.BboxParams(format='pascal_voc', label_fields=['category_ids']))

full_train = FashionInstanceDataset('train', transform=train_tf)
train_idx = np.random.choice(len(full_train), int(len(full_train)*FRACTION), replace=False)
train_loader = DataLoader(Subset(full_train, train_idx), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)

In [ ]:
def get_scratch_model(num_classes):
    # weights=None initializes the entire architecture from scratch
    model = torchvision.models.detection.maskrcnn_resnet50_fpn(weights="imagenet")
    
    # Replace Box Head
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Replace Mask Head
    in_features_mask = model.roi_heads.mask_predictor.conv5_mask.in_channels
    model.roi_heads.mask_predictor = MaskRCNNPredictor(in_features_mask, 256, num_classes)
    
    return model

model = get_scratch_model(NUM_CLASSES).to(device)

Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 184MB/s]


In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4) # Higher LR for scratch
scaler = torch.amp.GradScaler()

model.train()
for epoch in range(EPOCHS):
    epoch_loss = 0
    pbar = tqdm(train_loader, desc=f"Scratch Epoch {epoch+1}/{EPOCHS}")
    
    for images, targets in pbar:
        images = [img.to(device) for img in images]
        targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
        
        with torch.amp.autocast(device.type):
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
        
        optimizer.zero_grad()
        scaler.scale(losses).backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += losses.item()
        pbar.set_postfix(loss=f"{losses.item():.4f}")

torch.save(model.state_dict(), "maskrcnn_scratch_best.pth")

Scratch Epoch 5/5: 100%|██████████| 9011/9011 [1:18:02<00:00,  1.92it/s, loss=0.3507]


In [6]:
model.eval()
all_miou, all_dice = [], []

with torch.no_grad():
    # Evaluate on a 500-image subset for efficiency
    eval_idx = np.random.choice(len(full_train), 500)
    eval_loader = DataLoader(Subset(full_train, eval_idx), batch_size=2, collate_fn=collate_fn)
    
    for images, targets in tqdm(eval_loader, desc="Evaluating"):
        images = [img.to(device) for img in images]
        outputs = model(images)
        
        for i, output in enumerate(outputs):
            if len(output['masks']) == 0: continue
            
            # Binary thresholding for masks
            pred_mask = (output['masks'] > 0.5).squeeze(1).sum(dim=0).clamp(0, 1).cpu().numpy()
            gt_mask = targets[i]['masks'].sum(dim=0).clamp(0, 1).cpu().numpy()
            
            intersection = np.logical_and(pred_mask, gt_mask).sum()
            union = np.logical_or(pred_mask, gt_mask).sum()
            iou = intersection / (union + 1e-7)
            
            all_miou.append(iou)
            all_dice.append((2 * iou) / (1 + iou))

print(f"\n--- MASK R-CNN SCRATCH RESULTS ---")
print(f"Mean IoU: {np.mean(all_miou):.4f}")
print(f"Dice Score: {np.mean(all_dice):.4f}")

Evaluating: 100%|██████████| 250/250 [01:06<00:00,  3.74it/s]


--- MASK R-CNN SCRATCH RESULTS ---
Mean IoU: 0.7268
Dice Score: 0.8108
